In [1]:
import os
import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.dummy import DummyClassifier
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    average_precision_score,
    confusion_matrix,
    classification_report
)

print("Libraries imported successfully.")

Libraries imported successfully.


Load the corrected master dataset

In [2]:
BASE_DIR = os.path.dirname(os.path.dirname(os.path.abspath(os.getcwd())))

# If the path above does not work in your VS Code setup,
# use the direct project path instead.

MASTER_FILE = os.path.join(
    BASE_DIR,
    "LankaLandslide-AI",
    "data",
    "processed",
    "landslide_weather_master.csv"
)

df = pd.read_csv(MASTER_FILE)

print("Dataset shape:", df.shape)
print("\nColumns:")
print(df.columns.tolist())

Dataset shape: (192840, 14)

Columns:
['date', 'rainfall_mm', 'rain_mm', 'rain_hours', 'temp_max', 'temp_min', 'temp_mean', 'wind_max', 'wind_gust_max', 'district', 'latitude', 'longitude', 'landslide_events', 'landslide_recorded']


In [3]:
df["district"] = df["district"].replace({
    "Moneragala": "Monaragala"
})

print("Number of districts:", df["district"].nunique())
print(sorted(df["district"].unique()))

Number of districts: 24
['Ampara', 'Anuradhapura', 'Badulla', 'Batticaloa', 'Colombo', 'Galle', 'Gampaha', 'Hambantota', 'Jaffna', 'Kalutara', 'Kandy', 'Kegalle', 'Kilinochchi', 'Kurunegala', 'Mannar', 'Matale', 'Matara', 'Monaragala', 'Nuwara Eliya', 'Polonnaruwa', 'Puttalam', 'Ratnapura', 'Trincomalee', 'Vavuniya']


Create the target

In [4]:
TARGET = "landslide_recorded"

df[TARGET] = df[TARGET].astype(int)

print("Target distribution:")
print(df[TARGET].value_counts())

print("\nTarget percentage:")
print(
    (df[TARGET].value_counts(normalize=True) * 100).round(2)
)

Target distribution:
landslide_recorded
0    191996
1       844
Name: count, dtype: int64

Target percentage:
landslide_recorded
0    99.56
1     0.44
Name: proportion, dtype: float64


Demonstrate the baseline model trap

In [5]:
y = df[TARGET]

baseline = DummyClassifier(
    strategy="most_frequent"
)

baseline.fit(np.zeros((len(y), 1)), y)

baseline_pred = baseline.predict(
    np.zeros((len(y), 1))
)

print("Baseline model results")
print("----------------------")

print("Accuracy :", round(
    accuracy_score(y, baseline_pred), 4
))

print("Precision:", round(
    precision_score(y, baseline_pred, zero_division=0), 4
))

print("Recall   :", round(
    recall_score(y, baseline_pred, zero_division=0), 4
))

print("F1-score :", round(
    f1_score(y, baseline_pred, zero_division=0), 4
))

Baseline model results
----------------------
Accuracy : 0.9956
Precision: 0.0
Recall   : 0.0
F1-score : 0.0


Create rainfall features

In [6]:
df["date"] = pd.to_datetime(df["date"])

df = df.sort_values(
    ["district", "date"]
).reset_index(drop=True)

print("Data sorted successfully.")

Data sorted successfully.


Create lagged rainfall features

In [7]:
# Previous-day rainfall
df["rainfall_1day"] = (
    df.groupby("district")["rainfall_mm"]
      .shift(1)
)

# Rolling accumulated rainfall
for window in [3, 7, 14, 30]:
    df[f"rainfall_{window}day"] = (
        df.groupby("district")["rainfall_mm"]
          .transform(
              lambda x: x.shift(1).rolling(window).sum()
          )
    )

print(
    df[
        [
            "district",
            "date",
            "rainfall_mm",
            "rainfall_1day",
            "rainfall_3day",
            "rainfall_7day",
            "rainfall_14day",
            "rainfall_30day"
        ]
    ].head(15)
)

   district       date  rainfall_mm  rainfall_1day  rainfall_3day  \
0    Ampara 1998-01-01         18.0            NaN            NaN   
1    Ampara 1998-01-02          8.4           18.0            NaN   
2    Ampara 1998-01-03         20.9            8.4            NaN   
3    Ampara 1998-01-04         11.0           20.9           47.3   
4    Ampara 1998-01-05          0.7           11.0           40.3   
5    Ampara 1998-01-06          1.9            0.7           32.6   
6    Ampara 1998-01-07          2.9            1.9           13.6   
7    Ampara 1998-01-08          4.1            2.9            5.5   
8    Ampara 1998-01-09          4.3            4.1            8.9   
9    Ampara 1998-01-10          2.7            4.3           11.3   
10   Ampara 1998-01-11          0.1            2.7           11.1   
11   Ampara 1998-01-12          2.0            0.1            7.1   
12   Ampara 1998-01-13          0.3            2.0            4.8   
13   Ampara 1998-01-14          0.

Add rainfall intensity

In [8]:
df["rainfall_3day_avg"] = (
    df["rainfall_3day"] / 3
)

df["rainfall_7day_avg"] = (
    df["rainfall_7day"] / 7
)

df["rainfall_14day_avg"] = (
    df["rainfall_14day"] / 14
)

df["rainfall_30day_avg"] = (
    df["rainfall_30day"] / 30
)

print("Rainfall accumulation features created.")

Rainfall accumulation features created.


Temperature features

In [9]:
df["temperature_range"] = (
    df["temp_max"] - df["temp_min"]
)

print(
    df[
        [
            "temp_max",
            "temp_min",
            "temperature_range"
        ]
    ].head()
)

   temp_max  temp_min  temperature_range
0      27.2      23.5                3.7
1      27.8      23.6                4.2
2      26.1      23.6                2.5
3      26.1      23.5                2.6
4      28.2      24.6                3.6


Seasonal features

In [10]:
df["month"] = df["date"].dt.month

df["day_of_year"] = (
    df["date"].dt.dayofyear
)

# Cyclic encoding
df["month_sin"] = np.sin(
    2 * np.pi * df["month"] / 12
)

df["month_cos"] = np.cos(
    2 * np.pi * df["month"] / 12
)

print(
    df[
        [
            "date",
            "month",
            "month_sin",
            "month_cos"
        ]
    ].head()
)

        date  month  month_sin  month_cos
0 1998-01-01      1        0.5   0.866025
1 1998-01-02      1        0.5   0.866025
2 1998-01-03      1        0.5   0.866025
3 1998-01-04      1        0.5   0.866025
4 1998-01-05      1        0.5   0.866025


Add rainfall threshold indicators

In [11]:
df["heavy_rain_10mm"] = (
    df["rainfall_1day"] >= 10
).astype(int)

df["heavy_rain_25mm"] = (
    df["rainfall_1day"] >= 25
).astype(int)

df["heavy_rain_50mm"] = (
    df["rainfall_1day"] >= 50
).astype(int)

df["heavy_rain_100mm"] = (
    df["rainfall_1day"] >= 100
).astype(int)

print(
    df[
        [
            "rainfall_1day",
            "heavy_rain_10mm",
            "heavy_rain_25mm",
            "heavy_rain_50mm",
            "heavy_rain_100mm"
        ]
    ].head(10)
)

   rainfall_1day  heavy_rain_10mm  heavy_rain_25mm  heavy_rain_50mm  \
0            NaN                0                0                0   
1           18.0                1                0                0   
2            8.4                0                0                0   
3           20.9                1                0                0   
4           11.0                1                0                0   
5            0.7                0                0                0   
6            1.9                0                0                0   
7            2.9                0                0                0   
8            4.1                0                0                0   
9            4.3                0                0                0   

   heavy_rain_100mm  
0                 0  
1                 0  
2                 0  
3                 0  
4                 0  
5                 0  
6                 0  
7                 0  
8                 0 

Remove rows where rolling features aren't available

In [12]:
required_features = [
    "rainfall_1day",
    "rainfall_3day",
    "rainfall_7day",
    "rainfall_14day",
    "rainfall_30day"
]

before = len(df)

df = df.dropna(
    subset=required_features
).reset_index(drop=True)

after = len(df)

print("Rows before:", before)
print("Rows after :", after)
print("Rows removed:", before - after)

Rows before: 192840
Rows after : 192120
Rows removed: 720


Check the target again

In [13]:
print("Final target distribution:")
print(df[TARGET].value_counts())

print("\nPercentages:")
print(
    (df[TARGET].value_counts(normalize=True) * 100).round(3)
)

Final target distribution:
landslide_recorded
0    191278
1       842
Name: count, dtype: int64

Percentages:
landslide_recorded
0    99.562
1     0.438
Name: proportion, dtype: float64


Define the feature columns

In [14]:
FEATURES = [
    "rainfall_1day",
    "rainfall_3day",
    "rainfall_7day",
    "rainfall_14day",
    "rainfall_30day",

    "rainfall_3day_avg",
    "rainfall_7day_avg",
    "rainfall_14day_avg",
    "rainfall_30day_avg",

    "rain_hours",
    "temp_max",
    "temp_min",
    "temp_mean",
    "temperature_range",

    "wind_max",
    "wind_gust_max",

    "month_sin",
    "month_cos",

    "heavy_rain_10mm",
    "heavy_rain_25mm",
    "heavy_rain_50mm",
    "heavy_rain_100mm"
]

X = df[FEATURES]
y = df[TARGET]

print("Number of features:", len(FEATURES))
print("\nFeatures:")
for feature in FEATURES:
    print("-", feature)

Number of features: 22

Features:
- rainfall_1day
- rainfall_3day
- rainfall_7day
- rainfall_14day
- rainfall_30day
- rainfall_3day_avg
- rainfall_7day_avg
- rainfall_14day_avg
- rainfall_30day_avg
- rain_hours
- temp_max
- temp_min
- temp_mean
- temperature_range
- wind_max
- wind_gust_max
- month_sin
- month_cos
- heavy_rain_10mm
- heavy_rain_25mm
- heavy_rain_50mm
- heavy_rain_100mm


Save the engineered dataset

In [15]:
FEATURE_OUTPUT = os.path.join(
    os.path.dirname(MASTER_FILE),
    "landslide_weather_features.csv"
)

df.to_csv(
    FEATURE_OUTPUT,
    index=False
)

print("Feature dataset saved:")
print(FEATURE_OUTPUT)

print("\nFinal shape:", df.shape)

Feature dataset saved:
c:\Users\User\Documents\GitHub\LankaLandslide-AI\data\processed\landslide_weather_features.csv

Final shape: (192120, 32)


Check the engineered dataset

In [16]:
print("Dataset shape:", df.shape)

print("\nDate range:")
print(df["date"].min(), "to", df["date"].max())

print("\nTarget distribution:")
print(df[TARGET].value_counts())

print("\nTarget percentage:")
print(
    (df[TARGET].value_counts(normalize=True) * 100).round(3)
)

Dataset shape: (192120, 32)

Date range:
1998-01-31 00:00:00 to 2019-12-31 00:00:00

Target distribution:
landslide_recorded
0    191278
1       842
Name: count, dtype: int64

Target percentage:
landslide_recorded
0    99.562
1     0.438
Name: proportion, dtype: float64


Time-based split

In [17]:
train_df = df[
    df["date"] < "2016-01-01"
].copy()

val_df = df[
    (df["date"] >= "2016-01-01") &
    (df["date"] < "2018-01-01")
].copy()

test_df = df[
    df["date"] >= "2018-01-01"
].copy()

print("TRAIN:", train_df.shape)
print("VALIDATION:", val_df.shape)
print("TEST:", test_df.shape)

TRAIN: (157056, 32)
VALIDATION: (17544, 32)
TEST: (17520, 32)


Check class distribution in each split

In [18]:
def show_distribution(name, data):
    print(f"\n{name}")
    print("-" * 40)

    counts = data[TARGET].value_counts()
    percentages = (
        data[TARGET]
        .value_counts(normalize=True) * 100
    )

    print("No landslide:", counts.get(0, 0))
    print("Landslide   :", counts.get(1, 0))

    print(
        f"No landslide %: {percentages.get(0, 0):.3f}%"
    )
    print(
        f"Landslide %   : {percentages.get(1, 0):.3f}%"
    )


show_distribution("TRAIN", train_df)
show_distribution("VALIDATION", val_df)
show_distribution("TEST", test_df)


TRAIN
----------------------------------------
No landslide: 156272
Landslide   : 784
No landslide %: 99.501%
Landslide %   : 0.499%

VALIDATION
----------------------------------------
No landslide: 17529
Landslide   : 15
No landslide %: 99.915%
Landslide %   : 0.085%

TEST
----------------------------------------
No landslide: 17477
Landslide   : 43
No landslide %: 99.755%
Landslide %   : 0.245%


Create X/y datasets

In [19]:
X_train = train_df[FEATURES]
y_train = train_df[TARGET]

X_val = val_df[FEATURES]
y_val = val_df[TARGET]

X_test = test_df[FEATURES]
y_test = test_df[TARGET]

print("X_train:", X_train.shape)
print("y_train:", y_train.shape)

print("X_val:", X_val.shape)
print("y_val:", y_val.shape)

print("X_test:", X_test.shape)
print("y_test:", y_test.shape)

X_train: (157056, 22)
y_train: (157056,)
X_val: (17544, 22)
y_val: (17544,)
X_test: (17520, 22)
y_test: (17520,)


Baseline model on the test period

In [20]:
from sklearn.dummy import DummyClassifier

baseline = DummyClassifier(
    strategy="most_frequent"
)

baseline.fit(X_train, y_train)

baseline_pred = baseline.predict(X_test)

print("BASELINE MODEL")
print("=" * 50)

print(
    "Accuracy :",
    round(accuracy_score(y_test, baseline_pred), 4)
)

print(
    "Precision:",
    round(
        precision_score(
            y_test,
            baseline_pred,
            zero_division=0
        ),
        4
    )
)

print(
    "Recall   :",
    round(
        recall_score(
            y_test,
            baseline_pred,
            zero_division=0
        ),
        4
    )
)

print(
    "F1-score :",
    round(
        f1_score(
            y_test,
            baseline_pred,
            zero_division=0
        ),
        4
    )
)

BASELINE MODEL


Accuracy : 0.9975
Precision: 0.0
Recall   : 0.0
F1-score : 0.0


Train Logistic Regression with class weighting

In [21]:
from sklearn.linear_model import LogisticRegression

logistic_model = LogisticRegression(
    class_weight="balanced",
    max_iter=2000,
    random_state=42
)

logistic_model.fit(
    X_train,
    y_train
)

print("Logistic Regression trained successfully.")

Logistic Regression trained successfully.


C:\Users\User\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\LocalCache\local-packages\Python311\site-packages\sklearn\linear_model\_logistic.py:473: ConvergenceWarning: lbfgs failed to converge after 2000 iteration(s) (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT

Increase the number of iterations to improve the convergence (max_iter=2000).
You might also want to scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(


Logistic Regression predictions

In [22]:
logistic_pred = logistic_model.predict(X_test)

logistic_prob = logistic_model.predict_proba(
    X_test
)[:, 1]

print("Predictions generated.")

Predictions generated.


Evaluate Logistic Regression

In [23]:
logistic_accuracy = accuracy_score(
    y_test,
    logistic_pred
)

logistic_precision = precision_score(
    y_test,
    logistic_pred,
    zero_division=0
)

logistic_recall = recall_score(
    y_test,
    logistic_pred,
    zero_division=0
)

logistic_f1 = f1_score(
    y_test,
    logistic_pred,
    zero_division=0
)

logistic_roc_auc = roc_auc_score(
    y_test,
    logistic_prob
)

logistic_pr_auc = average_precision_score(
    y_test,
    logistic_prob
)

print("LOGISTIC REGRESSION")
print("=" * 50)

print(f"Accuracy : {logistic_accuracy:.4f}")
print(f"Precision: {logistic_precision:.4f}")
print(f"Recall   : {logistic_recall:.4f}")
print(f"F1-score : {logistic_f1:.4f}")
print(f"ROC-AUC  : {logistic_roc_auc:.4f}")
print(f"PR-AUC   : {logistic_pr_auc:.4f}")

LOGISTIC REGRESSION
Accuracy : 0.7901
Precision: 0.0105
Recall   : 0.9070
F1-score : 0.0208
ROC-AUC  : 0.9170
PR-AUC   : 0.0371


Confusion matrix

In [24]:
cm = confusion_matrix(
    y_test,
    logistic_pred
)

print("Confusion Matrix")
print(cm)

Confusion Matrix
[[13804  3673]
 [    4    39]]


In [25]:
print("\nClassification Report")
print(
    classification_report(
        y_test,
        logistic_pred,
        target_names=[
            "No Landslide",
            "Landslide"
        ],
        zero_division=0
    )
)


Classification Report
              precision    recall  f1-score   support

No Landslide       1.00      0.79      0.88     17477
   Landslide       0.01      0.91      0.02        43

    accuracy                           0.79     17520
   macro avg       0.51      0.85      0.45     17520
weighted avg       1.00      0.79      0.88     17520



Random Forest

In [26]:
from sklearn.ensemble import RandomForestClassifier

rf_model = RandomForestClassifier(
    n_estimators=300,
    class_weight="balanced",
    random_state=42,
    n_jobs=-1
)

rf_model.fit(
    X_train,
    y_train
)

print("Random Forest trained successfully.")

Random Forest trained successfully.


Random Forest predictions

In [27]:
rf_pred = rf_model.predict(X_test)

rf_prob = rf_model.predict_proba(
    X_test
)[:, 1]

print("Random Forest predictions generated.")

Random Forest predictions generated.


Evaluate Random Forest

In [28]:
rf_accuracy = accuracy_score(
    y_test,
    rf_pred
)

rf_precision = precision_score(
    y_test,
    rf_pred,
    zero_division=0
)

rf_recall = recall_score(
    y_test,
    rf_pred,
    zero_division=0
)

rf_f1 = f1_score(
    y_test,
    rf_pred,
    zero_division=0
)

rf_roc_auc = roc_auc_score(
    y_test,
    rf_prob
)

rf_pr_auc = average_precision_score(
    y_test,
    rf_prob
)

print("RANDOM FOREST")
print("=" * 50)

print(f"Accuracy : {rf_accuracy:.4f}")
print(f"Precision: {rf_precision:.4f}")
print(f"Recall   : {rf_recall:.4f}")
print(f"F1-score : {rf_f1:.4f}")
print(f"ROC-AUC  : {rf_roc_auc:.4f}")
print(f"PR-AUC   : {rf_pr_auc:.4f}")

RANDOM FOREST
Accuracy : 0.9975
Precision: 0.0000
Recall   : 0.0000
F1-score : 0.0000
ROC-AUC  : 0.8084
PR-AUC   : 0.0242


Compare the models

In [29]:
results = pd.DataFrame({
    "Model": [
        "Baseline",
        "Logistic Regression",
        "Random Forest"
    ],
    "Accuracy": [
        accuracy_score(y_test, baseline_pred),
        logistic_accuracy,
        rf_accuracy
    ],
    "Precision": [
        precision_score(
            y_test,
            baseline_pred,
            zero_division=0
        ),
        logistic_precision,
        rf_precision
    ],
    "Recall": [
        recall_score(
            y_test,
            baseline_pred,
            zero_division=0
        ),
        logistic_recall,
        rf_recall
    ],
    "F1": [
        f1_score(
            y_test,
            baseline_pred,
            zero_division=0
        ),
        logistic_f1,
        rf_f1
    ],
    "ROC_AUC": [
        np.nan,
        logistic_roc_auc,
        rf_roc_auc
    ],
    "PR_AUC": [
        np.nan,
        logistic_pr_auc,
        rf_pr_auc
    ]
})

results

,Model,Accuracy,Precision,Recall,F1,ROC_AUC,PR_AUC
0,Baseline,0.997546,0.000000,0.000000,0.000000,NaN,NaN
1,Logistic Regression,0.790126,0.010506,0.906977,0.020772,0.916951,0.03710
2,Random Forest,0.997546,0.000000,0.000000,0.000000,0.808372,0.02416


Save the trained models

In [30]:
import joblib

MODELS_DIR = os.path.join(
    os.path.dirname(os.path.dirname(MASTER_FILE)),
    "..",
    "models"
)

MODELS_DIR = os.path.abspath(MODELS_DIR)

os.makedirs(
    MODELS_DIR,
    exist_ok=True
)

print("Models folder:")
print(MODELS_DIR)

Models folder:
c:\Users\User\Documents\GitHub\LankaLandslide-AI\models


In [31]:
joblib.dump(
    logistic_model,
    os.path.join(
        MODELS_DIR,
        "logistic_regression_balanced.pkl"
    )
)

joblib.dump(
    rf_model,
    os.path.join(
        MODELS_DIR,
        "random_forest_balanced.pkl"
    )
)

print("Models saved successfully.")

Models saved successfully.
